In [ ]:
#----------------------------------------------
# EXERCITIUL ASTA NU ARE SENS SA IL CORECTATI
# am strict cat am mai avut timp si cat am stiut si din ce materiale am mai avut
# sper sa se puncteze notiunile
#---------------------------------------------

import os
import cv2
import numpy as np

# Paths to image directories and output
IMAGE_DIR = "Subiect3/data/ex2"
OUTPUT_DIR = "Subiect3/data/ex2/results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Board and cell sizes (pixels)
BOARD_SIZE = 600            # final warped board will be BOARD_SIZE x BOARD_SIZE
CELL_SIZE = BOARD_SIZE // 3

# HSV color thresholds for detecting the board (e.g. black grid lines on a white board)
# (These are placeholders; adjust depending on actual board colors)
LOWER_GRID = np.array([0, 0, 0])
UPPER_GRID = np.array([180, 255, 50])

# HSV color thresholds for X and O marks (example: X in red, O in green)
LOWER_NUM = np.array([0, 0, 0]); UPPER_NUM = np.array([180, 255, 50])



In [ ]:
def order_points(pts):
  
    rect = np.zeros((4, 2), dtype="float32")
    s = pts.sum(axis=1)
    diff = np.diff(pts, axis=1)
    rect[0] = pts[np.argmin(s)]
    rect[2] = pts[np.argmax(s)]
    rect[1] = pts[np.argmin(diff)]
    rect[3] = pts[np.argmax(diff)]
    return rect

def find_and_warp_board(image):
    
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    mask = cv2.inRange(hsv, LOWER_GRID, UPPER_GRID)
    # Find contours
    cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cnts:
        return None
    # Select the largest contour with four corners
    cnts = sorted(cnts, key=cv2.contourArea, reverse=True)
    board_contour = None
    for cnt in cnts:
        peri = cv2.arcLength(cnt, True)
        approx = cv2.approxPolyDP(cnt, 0.02 * peri, True)
        if len(approx) == 4:
            board_contour = approx
            break
    if board_contour is None:
        return None
    pts = board_contour.reshape(4, 2)
    rect = order_points(pts)
    dst = np.array([[0, 0], [BOARD_SIZE - 1, 0], [BOARD_SIZE - 1, BOARD_SIZE - 1], [0, BOARD_SIZE - 1]], dtype="float32")
    M = cv2.getPerspectiveTransform(rect, dst)
    warp = cv2.warpPerspective(image, M, (BOARD_SIZE, BOARD_SIZE))
    return warp


In [ ]:
def classify_cell(cell):
    """
    Classify a cell as 'X', 'O', or ' ' (empty) based on color thresholding.
    """
    hsv = cv2.cvtColor(cell, cv2.COLOR_BGR2HSV)

    mask_num = cv2.inRange(hsv,LOWER_NUM,UPPER_NUM)
    # Count thresholded pixels for each symbol
    count_x = cv2.countNonZero(mask_x)
    count_o = cv2.countNonZero(mask_o)
    # Decide symbol
    if count_x > count_o and count_x > 50:  # threshold to avoid noise
        return 'X'
    elif count_o > count_x and count_o > 50:
        return 'O'
    else:
        return ' '


In [ ]:
for fname in os.listdir(IMAGE_DIR):
    img_path = os.path.join(IMAGE_DIR, fname)
    image = cv2.imread(img_path)
    if image is None:
        continue
    # Detect and warp board
    board = find_and_warp_board(image)
    if board is None:
        print(f"Board not found in {fname}, skipping.")
        continue
    # Classify each cell
    result_rows = []
    for i in range(3):
        row_symbols = []
        for j in range(3):
            y1, y2 = i * CELL_SIZE, (i + 1) * CELL_SIZE
            x1, x2 = j * CELL_SIZE, (j + 1) * CELL_SIZE
            cell = board[y1:y2, x1:x2]
            symbol = classify_cell(cell)
            row_symbols.append(symbol)
        result_rows.append(" ".join(row_symbols))
    # Save the 3x3 grid to a text file
    base_name = os.path.splitext(fname)[0]
    out_path = os.path.join(OUTPUT_DIR, base_name + ".txt")
    with open(out_path, "w") as f:
        for row in result_rows:
            f.write(row + "\n")
